In [9]:
from llama_index.readers.file import PDFReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings, VectorStoreIndex, Document
from pathlib import Path
import pandas as pd


In [10]:
CSV_PATH = Path('../Data/ChatbotData.csv')
ollama_base_url = 'http://localhost:11434'

Settings.llm = Ollama(
    model='gemma4:e4b',
    base_url=ollama_base_url,
    request_timeout=120.0,
    temperature = 0 # 낮을수록 좋은거. gpt 기준 0.7이 일반, 0.5가 plus, 0.2가 pro 정도 된다고 함  

)
Settings.embed_model = OllamaEmbedding(
    model_name='nomic-embed-text',
    base_url=ollama_base_url,
    request_timeout=120.0
)

print(f"CSV경로: {CSV_PATH.resolve()}")
print(f"Ollama / LlamaIndex 설정 완료")

CSV경로: /Users/mac/Documents/DeepLearning/RAG/Data/ChatbotData.csv
Ollama / LlamaIndex 설정 완료


#### CSV를 문서 형태로 변환

In [11]:
df = pd.read_csv(CSV_PATH)
df.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [12]:
TEXT_COLUMNS = ['Q','A']
METADATA_COLUMNS = ['label']

# MAX_ROWS = 1000
# df = df.head(MAX_ROWS).copy()

display(df.head())
print('문서와 대상 컬럼 : ',TEXT_COLUMNS)
print('메타데이터 컬럼 : ',METADATA_COLUMNS)
# print('사용할 행수 : ',MAX_ROWS)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


문서와 대상 컬럼 :  ['Q', 'A']
메타데이터 컬럼 :  ['label']


In [13]:
# 각 Row를 질문-답변 형태의 문서로 변환

def row_to_document(row:pd.Series, row_number:int) -> Document:
    text_parts = []

    for column in TEXT_COLUMNS:
        value = row[column]

        if pd.isna(value):
            continue
        text_parts.append(f'{column}:{value}')

    metadata = {
        'row_number' : row_number,
        'label' : row['label']
    }
    return Document(
        text = ' | '.join(text_parts),
        metadata = metadata
    )    
# DataFrame의 각 row를 Document로 변환
documents = [row_to_document(row, idx) for idx, row in df.iterrows()]

print('생성된 Document수 : ',len(documents))
print('첫번째 Document 예제')
print(documents[0].text)

생성된 Document수 :  11823
첫번째 Document 예제
Q:12시 땡! | A:하루가 또 가네요.


In [14]:
# 문서목록으로 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

# 검색된 문서를 바탕으로 답변하는 chat engine을 제작
# as_query_engine는 검색하여 답변
# as_chat_engine은 추론
chat_engine = index.as_chat_engine(
    chat_mode='context',
    similartity_top_k = 5,
    verbose = True
)

2026-04-28 13:55:13,899 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,120 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,336 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,463 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,585 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,791 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:14,930 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:15,073 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:15,210 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:55:15,481 - INFO - HTTP Request: POST http://localhost:1143

In [15]:
# Test
question = '12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?'
response = chat_engine.chat(question)
print('질문:',question)
print('응답:',response)

2026-04-28 13:58:51,753 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 13:59:18,775 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


질문: 12시 땡! 이라는 질문에는 어떤 답변이 연결되어 있어?
응답: '12시 땡!'이라는 질문에는 **"하루가 또 가네요."**라는 답변이 연결되어 있습니다.


In [ ]:
# 여러번 질문하고
# exit, quit 종료
while True:
    user_question = input('질문을 입력하세요:').strip()
    if user_question.lower() in {'exit', 'quit'}:
        print('쳇봇을 종료합니다.')
        break
    if not user_question:
        print('빈 질문은 처리 할 수 없다. 다시 입력하여라')
        continue

    answer = chat_engine.chat(user_question)
    print('\n[응답]')
    print(answer)
    print('-'*50)    


2026-04-28 14:03:10,236 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 14:03:14,871 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



[응답]
안녕하세요! 😊 무엇을 도와드릴까요?
--------------------------------------------------


2026-04-28 14:03:29,986 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-28 14:04:14,939 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"



[응답]
혹시 다른 궁금한 점이나 제가 도와드릴 내용이 있으신가요? 😊 편하게 말씀해주세요!
--------------------------------------------------


2026-04-28 14:04:49,570 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
